# EDA 02 — McAuley Metadata (35M Products)
**Purpose:** Shape check and schema discovery of the McAuley Lab Amazon metadata.  
**Data:** 33 category CSVs, 37.5 GB total  
**Engine:** DuckDB (direct CSV query)

In [ ]:
import duckdb
import os

con = duckdb.connect()

files = []
folder = '../collection/mcauley_csv'
for f in sorted(os.listdir(folder)):
    if f.endswith('.csv'):
        size_mb = os.path.getsize(os.path.join(folder, f)) / (1024 * 1024)
        files.append((f, round(size_mb, 1)))

print(f"Total files: {len(files)}")
print(f"Total size: {sum(s for _, s in files):.1f} MB\n")
for name, size in files:
    print(f"  {name:45s} {size:>8.1f} MB")

### Finding: 33 Categories, 37.5 GB
Books (7.2 GB) and Clothing (6.2 GB) dominate — together they're 36% of all metadata. Smallest: Subscription Boxes (0.5 MB), Gift Cards (0.8 MB). This matches the 33 McAuley top-level categories vs Kaggle's 248 subcategories — reconciling these hierarchies is a Silver layer task.

In [ ]:
con.sql("""
    SELECT *
    FROM read_csv_auto('../collection/mcauley_csv/meta_All_Beauty.csv')
    LIMIT 3
""").show(max_width=200)

### Schema: 12 Columns
parent_asin, main_category, title, average_rating, rating_number, price, store, features, description, categories, details_brand, details_manufacturer. Note: many NULLs visible — need to quantify across all 33 files.

In [ ]:
con.sql("""
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN price IS NULL OR price = '' OR price = '—' THEN 1 ELSE 0 END) as null_price,
        SUM(CASE WHEN store IS NULL OR store = '' THEN 1 ELSE 0 END) as null_store,
        SUM(CASE WHEN features IS NULL OR features = '' THEN 1 ELSE 0 END) as null_features,
        SUM(CASE WHEN description IS NULL OR description = '' THEN 1 ELSE 0 END) as null_desc,
        SUM(CASE WHEN categories IS NULL OR categories = '' THEN 1 ELSE 0 END) as null_categories,
        SUM(CASE WHEN details_brand IS NULL OR details_brand = '' THEN 1 ELSE 0 END) as null_brand,
        SUM(CASE WHEN details_manufacturer IS NULL OR details_manufacturer = '' THEN 1 ELSE 0 END) as null_manufacturer
    FROM read_csv_auto('../collection/mcauley_csv/meta_Electronics.csv', types={'price': 'VARCHAR'})
""").show()

### Finding: McAuley Null Rates (Electronics, 1.6M rows)
- **Price:** 67% null/invalid — most products have no price. Will need Kaggle price as primary.
- **Store:** Near-complete (only 2 nulls) — strong for seller analysis.
- **Features:** 26% null — partial coverage.
- **Description:** 42% null — nearly half have no description.
- **Categories:** 8% null — good coverage.
- **Brand:** 28% null — workable, especially combined with store.
- **Manufacturer:** 15% null — solid.

Key takeaway: McAuley's value is store/brand/manufacturer (high coverage), NOT price. Kaggle brings price + sales. The LEFT JOIN in Silver is essential.

In [ ]:
import os

results = []
folder = '../collection/mcauley_csv'
for f in sorted(os.listdir(folder)):
    if f.endswith('.csv'):
        cat = f.replace('meta_', '').replace('.csv', '')
        count = con.sql(f"""
            SELECT COUNT(*) as n 
            FROM read_csv_auto('{folder}/{f}', types={{'price': 'VARCHAR'}})
        """).fetchone()[0]
        results.append((cat, count))
        print(f"  {cat:40s} {count:>10,}")

total = sum(n for _, n in results)
print(f"\n  {'TOTAL':40s} {total:>10,}")

### Finding: 35M Products Across 33 Categories
Clothing (7.2M), Books (4.4M), and Home & Kitchen (3.7M) are the top 3 — together 44% of all metadata. Smallest: Subscription Boxes (641), Gift Cards (1,137), Magazine Subscriptions (3,391). Note: Health_and_Personal_Care (60K) exists alongside Health_and_Household (798K) — possible overlap to investigate in Silver.

## Summary
- **35M products** across 33 top-level categories (vs Kaggle's 248 subcategories)
- **12 columns:** parent_asin, main_category, title, average_rating, rating_number, price, store, features, description, categories, details_brand, details_manufacturer
- **Price is 67%+ null** — McAuley is NOT a price source. Kaggle brings price + sales.
- **Store near-complete** (99.99%) — this is the seller analysis goldmine
- **Brand/manufacturer solid** (72-85% coverage)
- **Em-dash "—" used as null for price** — must cast as VARCHAR and clean in Bronze
- **Top 3 categories = 44% of data** (Clothing, Books, Home & Kitchen)
- **Category overlap:** Health_and_Personal_Care vs Health_and_Household needs reconciliation

### Next: Bronze pipeline — load Kaggle + McAuley into DuckDB with quality flags